In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd

df = pd.read_csv(path + "/" + "Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
import matplotlib.pyplot as plt


def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop('Order_ID', axis=1)

In [ ]:
# Task 2: Write your code here:

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# since the missing values are small compared to the data, we can drop them
df = df.dropna()

In [ ]:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Task 3: Write your code here:
# Do we have duplicate samples? drop if yes
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder

# Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

for col in categorical_cols:
  print(f"Encoding column: {col}")
  ohe = OneHotEncoder(sparse_output=False)
  # Apply fit_transform to one hot encode the column and join the resulting columns to the df
  df = df.join(
    pd.DataFrame(
        ohe.fit_transform(df[[col]]),
        columns=ohe.get_feature_names_out([col]),
        index=df.index
    )
    ).drop(columns=[col])

df.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

numerical_cols = df.select_dtypes(include=["number"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
df.describe()

In [ ]:
# Task 6: Write your code here:
check_target_distribution(df, "Delivery_Time") # the target column is similar to a normal distribution

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float) # don't include the target in X
y = df['Delivery_Time'].astype(float) # take only the target for y

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import numpy as np

model = RandomForestRegressor(n_estimators=200)

n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for linear regression results for each fold
lr_losses = []
lr_mae = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  print(f"Training...")

  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Calculate metrics
  mae = mean_absolute_error(y_test, y_pred)

  # Store results
  lr_mae.append(mae)

print(f"Average MAE score across all folds: {np.mean(lr_mae)}")

In [ ]:
# Is our model good
# Calculate the baseline predictions (mean of the target)
baseline_pred = np.full_like(y, y.mean())

# Evaluate the baseline
baseline_mae = mean_absolute_error(y, baseline_pred)

print(f"Baseline MAE (using mean target): {baseline_mae:.4f}")

In [ ]:
imp = model.feature_importances_

# Create a 1x3 plot
fig = plt.figure(figsize=(18, 6))
features = X.columns

model_name = "Random Forest"
  # Sort features by importance for a cleaner plot
sorted_idx = np.argsort(imp)

plt.barh(features[sorted_idx], imp[sorted_idx])
plt.title(f"{model_name} Feature Importance")
plt.xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
predictions = model.predict(X_test)
plt.figure(figsize=(8, 6))

# Plot: Predicted vs Actual scatter
plt.scatter(y_test, predictions, alpha=0.5, s=10, c='steelblue')

# Add perfect prediction line
min_val = min(y_test.min(), predictions.min())
max_val = max(y_test.max(), predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

plt.xlabel('Actual Price ($)', fontsize=12)
plt.ylabel('Predicted Price ($)', fontsize=12)
plt.title('Predicted vs Actual Diamond Prices', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
!pip install catboost -q

In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor

models = {
  "rfr": RandomForestRegressor(n_estimators=200),
  "cat": CatBoostRegressor(verbose=0)
}

n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for linear regression results for each fold
lr_losses = []
lr_mae = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  print(f"Training...")

  # Train
  models['cat'].fit(X_train, y_train)
  models['rfr'].fit(X_train, y_train)

  # Predict
  y_pred_cat = models['cat'].predict(X_test)
  y_pred_rfr = models['rfr'].predict(X_test)

  # Calculate metrics
  mae = mean_absolute_error(y_test, (y_pred_cat + y_pred_rfr) / 2)

  # Store results
  lr_mae.append(mae)

print(f"Average MAE score across all folds: {np.mean(lr_mae)}")